In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lower, regexp_replace, split, explode, year, row_number, desc, regexp_extract
from pyspark.sql.window import Window

# 1. Инициализация Spark сессии (исправлено)
spark = SparkSession.builder \
    .appName("Top 10 Programming Languages Report") \
    .getOrCreate()

# 2. Загрузка списка языков программирования
languages_raw = spark.read.csv("programming-languages.csv", header=True)
# Приводим к нижнему регистру для сопоставления с тегами
languages_list = languages_raw.select(
    lower(col("name")).alias("lang_name_lower"),
    col("name").alias("original_name")
)

# 3. Загрузка постов из XML
# Читаем как текст, чтобы распарсить атрибуты регулярками
posts_raw = spark.read.text("posts_sample.xml")

# Извлекаем CreationDate и Tags
posts_parsed = posts_raw.filter(col("value").contains("<row")) \
    .select(
        regexp_extract(col("value"), 'CreationDate="([^"]+)"', 1).alias("date_str"),
        regexp_extract(col("value"), 'Tags="([^"]+)"', 1).alias("tags_str")
    )

# 4. Обработка данных: фильтрация по годам и "развертывание" тегов
# Теги в формате &lt;java&gt;&lt;python&gt; преобразуем в список
posts_refined = posts_parsed.select(
    year(col("date_str")).alias("year"),
    regexp_replace(col("tags_str"), '^&lt;|&gt;$', '').alias("tags_cleaned")
).filter((col("year") >= 2010) & (col("year") <= 2020))

# Разбиваем строку тегов по разделителю &gt;&lt; и делаем explode
posts_exploded = posts_refined.withColumn("tag", explode(split(col("tags_cleaned"), "&gt;&lt;"))) \
    .withColumn("tag", lower(col("tag")))

# 5. Соединение со справочником языков и подсчет упоминаний
report_all = posts_exploded.join(
    languages_list,
    posts_exploded.tag == languages_list.lang_name_lower
).groupBy("year", "original_name") \
    .count() \
    .withColumnRenamed("original_name", "language") \
    .withColumnRenamed("count", "mentions")

# 6. Определение ТОП-10 для каждого года через оконную функцию
window_spec = Window.partitionBy("year").orderBy(desc("mentions"))

top10_report = report_all.withColumn("rank", row_number().over(window_spec)) \
    .filter(col("rank") <= 10) \
    .select("year", "language", "mentions") \
    .orderBy("year", desc("mentions"))

# 7. Вывод результата в консоль для проверки
top10_report.show(50, truncate=False)

# 8. Сохранение в формат Parquet
top10_report.write.mode("overwrite").parquet("top_10_languages_report.parquet")

print("Отчёт успешно сохранен в 'top_10_languages_report.parquet'")

+----+-----------+--------+
|year|language   |mentions|
+----+-----------+--------+
|2010|JavaScript |20      |
|2010|Java       |15      |
|2010|PHP        |15      |
|2010|C          |8       |
|2010|Objective-C|7       |
|2010|Python     |6       |
|2010|Ruby       |6       |
|2010|Delphi     |2       |
|2010|Bash       |2       |
|2010|Perl       |2       |
|2011|Java       |32      |
|2011|PHP        |25      |
|2011|JavaScript |19      |
|2011|Python     |11      |
|2011|C          |10      |
|2011|Objective-C|10      |
|2011|Perl       |6       |
|2011|Ruby       |5       |
|2011|ColdFusion |3       |
|2011|Delphi     |2       |
|2012|PHP        |51      |
|2012|JavaScript |47      |
|2012|Java       |40      |
|2012|Python     |25      |
|2012|Objective-C|12      |
|2012|Ruby       |11      |
|2012|C          |10      |
|2012|Curl       |3       |
|2012|Scala      |3       |
|2012|Bash       |3       |
|2013|JavaScript |53      |
|2013|Java       |51      |
|2013|PHP        |45